In [3]:
import tiktoken

# Текст для обработки
text = """
FastAPI — это современный, высокопроизводительный веб-фреймворк для построения API на Python, основанный на стандартах OpenAPI и JSON Schema. Он автоматически генерирует интерактивную документацию через Swagger UI и ReDoc, что значительно упрощает разработку и тестирование RESTful-сервисов. FastAPI полностью поддерживает асинхронное программирование с использованием async/await, что делает его идеальным выбором для высоконагруженных приложений и микросервисной архитектуры. Благодаря строгой валидации входных данных с помощью Pydantic, FastAPI помогает избегать ошибок на ранних этапах и обеспечивает надёжность типов даже в динамическом языке, таком как Python.

Alembic — это мощный инструмент для управления миграциями базы данных в экосистеме SQLAlchemy. Он позволяет безопасно обновлять схему базы данных при изменении моделей в коде, сохраняя целостность данных и историю изменений. Alembic поддерживает автогенерацию миграций на основе сравнения текущей модели с состоянием БД, а также ручное редактирование миграционных скриптов для сложных кейсов, таких как переименование колонок или перенос данных. В сочетании с FastAPI он часто используется в продакшене для обеспечения согласованности между ORM-моделями и реальной структурой базы данных.

Docker — это платформа для контейнеризации приложений, которая упрощает развёртывание, масштабирование и изоляцию зависимостей. С помощью Docker вы можете упаковать своё приложение, его зависимости и конфигурацию в единый образ, который будет работать одинаково на любом сервере, поддерживающем Docker. Это особенно полезно в CI/CD-пайплайнах, где важно избежать “у меня локально работает”-проблем. Docker Compose позволяет описывать многосервисные архитектуры (например, FastAPI + PostgreSQL + Redis) в одном YAML-файле, что ускоряет локальную разработку и тестирование.
"""

CHUNK_SIZE = 400

# Инициализация кодировщика
enc = tiktoken.get_encoding("cl100k_base")

# Простой чанкинг: собираем предложения, пока не превысим CHUNK_SIZE
sentences = [s.strip() + "." for s in text.split(".") if s.strip()]
chunks = []
current = ""

for sent in sentences:
    test_chunk = (current + " " + sent).strip() if current else sent
    if len(enc.encode(test_chunk)) <= CHUNK_SIZE:
        current = test_chunk
    else:
        if current:
            chunks.append(current)
        current = sent

if current:
    chunks.append(current)

# Вывод длины чанка в токенах
first_chunk_tokens = len(enc.encode(chunks[1]))
print(first_chunk_tokens)

378


In [6]:
def deduplicate_chunks(chunks: list[str]) -> list[str]:
    new_list = list()
    for chunk in chunks:
        if chunk not in new_list:
            new_list.append(chunk)
    return new_list

In [44]:
pip install pymorphy2 -q

Note: you may need to restart the kernel to use updated packages.


In [45]:
import re
from pymorphy2 import MorphAnalyzer

morph = MorphAnalyzer()

def _extract_lemmas(text: str) -> set[str]:
    """Извлекает значимые слова из текста и возвращает множество их лемм."""
    # 1. Разбиваем текст на слова, отсекая пунктуацию, цифры и спецсимволы
    words = re.findall(r'[a-zA-Zа-яА-ЯёЁ]+', text)
    
    # 2. Фильтруем слова короче 3 символов
    significant_words = [w for w in words if len(w) >= 3]
    
    # 3. Лемматизируем и собираем во множество
    return {morph.parse(w)[0].normal_form for w in significant_words}

def is_relevant(query: str, retrieved_chunk: str) -> bool:
    """Определяет релевантность чанка на основе совпадения лемм значимых слов."""
    query_lemmas = _extract_lemmas(query)
    chunk_lemmas = _extract_lemmas(retrieved_chunk)
    
    # 4. Возвращаем True, если пересечение множеств не пусто
    return len(query_lemmas & chunk_lemmas) > 0

/opt/anaconda3/envs/LLM_course/lib/python3.10/site-packages/pymorphy2/analyzer.py:114: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [46]:
is_relevant("дай все документации", "FastAPI автоматически генерирует документацию")

True

In [42]:
class RAGPipelineConfig:
    def __init__(self, chunk_size: int, embedding_model: str, vector_db: str, reranker_enabled: bool):
        if 100 <= chunk_size <= 1000 and embedding_model in ["all-MiniLM-L6-v2", "bge-m3"] and vector_db in ["chroma", "qdrant"]:
            self.chunk_size = chunk_size
            self.embedding_model = embedding_model
            self.vector_db = vector_db
            self.reranker_enabled = reranker_enabled
        else:
            raise ValueError

    def to_dict(self):
        return {
            "chunk_size": self.chunk_size, 
            "embedding_model": self.embedding_model, 
            "vector_db": self.vector_db, 
            "reranker_enabled": self.reranker_enabled}

    def is_production_ready(self):
        if self.vector_db=="qdrant":
            return True
        else:
            return False

In [43]:
rag = RAGPipelineConfig(105, "bge-m3", "qdrant", True)